<a href="https://colab.research.google.com/github/yoshiagent/IntControl-colab/blob/main/notebooks/GA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 開啟"/></a>

# 遺傳演算法（GA）：原課程 MATLAB 程式的 Python 版

這本筆記本把「智慧型控制」課程遺傳演算法章的 MATLAB 程式逐一改寫成 Python，並實際執行。對照的網頁：
[GA 章首](https://yoshiagent.github.io/IntControl/GA.html)、[初始族群](https://yoshiagent.github.io/IntControl/GA-GA_initPopu.html)、
[解碼](https://yoshiagent.github.io/IntControl/GA-decode.html)、[選擇](https://yoshiagent.github.io/IntControl/GA-Select.html)、
[交配](https://yoshiagent.github.io/IntControl/GA-crossover.html)、[突變](https://yoshiagent.github.io/IntControl/GA-GA_Mutate.html)、
[菁英保留](https://yoshiagent.github.io/IntControl/GA-Elitism.html)、[適應函數](https://yoshiagent.github.io/IntControl/GA-FitnessFunction.html)。

**使用方式**：在 Colab 選「執行階段 → 全部執行」，或由上往下逐格按 ▶。只用 Colab 內建的 numpy 與 plotly（互動圖，可縮放、滑鼠移上去看數值）。

**MATLAB 與 Python 的差別**（改寫時要注意的地方）：

| MATLAB | Python（numpy） |
|---|---|
| 索引從 1 開始，`popu(i,:)` | 索引從 0 開始，`popu[i, :]` |
| `a(k:n)` 含頭含尾 | `a[k-1:n]`，切片不含尾端 |
| `rand` | `rng.random()`（`rng` 是有固定種子的亂數產生器，結果可以重現） |
| `fix(x)` | `int(x)`（正數時相同） |
| `polyval(bits, 2)`（二進位轉十進位） | `np.polyval(bits, 2)` |
| `[s, ind] = sort(f)` | `ind = np.argsort(f, kind="stable")`、`s = f[ind]` |

MATLAB 與 numpy 的亂數序列不同，所以每次演化的過程和原講義的執行結果不會一模一樣，但演算法完全相同。

In [1]:
import re
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

rng = np.random.default_rng(1)          # 相當於 MATLAB 的 rand；換種子就是另一次執行

## 1. 七個 GA 函式

### GA1_InitPopu：產生初始族群

```matlab
function popu=GA1_InitPopu(popusize,bit_n,var_n)
%popusize為染色體總數
%bit_n 為每個變數之位元數
%var_n 為變數總數
popu=rand(popusize,bit_n*var_n)>0.5;
```

In [2]:
def GA1_InitPopu(popusize, bit_n, var_n):
    """每個位元以 0.5 的機率為 1；回傳 popusize × (bit_n·var_n) 的 0/1 陣列"""
    return (rng.random((popusize, bit_n * var_n)) > 0.5).astype(int)

popu = GA1_InitPopu(4, 8, 2)
print(popu)

[[1 1 0 1 0 0 1 0 1 0 1 1 0 1 0 0]
 [0 0 0 0 1 0 0 1 1 1 1 0 0 1 1 0]
 [1 1 1 1 0 1 0 0 1 1 1 0 1 1 1 1]
 [0 1 1 1 0 1 0 0 1 1 1 0 0 0 1 1]]


### GA2_Decode：解碼

```matlab
function result=GA2_Decode(popu,range)
   popusize=size(popu,1);       % get the size of population
   var_no=size(range,2);        % get the number of parameters
   n=size(popu,2)/var_no;       % calculate the bit number of each parameter.
   for j=1:popusize             % for each chromosome
      for i=1 : var_no          % for each parameter
         dec=polyval(popu(j,(i-1)*n+1:i*n),2);   % transform binary to decimal
         result(j,i)=range(1,i)+dec*(range(2,i)-range(1,i))/(2^n-1);  % mapping to specified range
      end
   end
```
`range` 是 2 × var_no 的矩陣：第一列是各變數的下限、第二列是上限。`polyval(bits, 2)` 把位元當成多項式的係數、代入 2，就是二進位轉十進位。

In [3]:
def GA2_Decode(popu, range_):
    popusize, var_no = popu.shape[0], range_.shape[1]
    n = popu.shape[1] // var_no                       # 每個參數的位元數
    result = np.zeros((popusize, var_no))
    for j in range(popusize):
        for i in range(var_no):
            bits = popu[j, i * n:(i + 1) * n]
            dec = np.polyval(bits, 2)                 # 和 MATLAB 一樣用 polyval 轉十進位
            result[j, i] = range_[0, i] + dec * (range_[1, i] - range_[0, i]) / (2 ** n - 1)
    return result

rng_ = np.array([[1, 1], [10, 10]])                    # 兩個變數都在 1～10
print(GA2_Decode(np.array([[0] * 8 + [1] * 8, [1, 0, 0, 0, 0, 0, 0, 0] * 2]), rng_))
# 第一個染色體：x 全 0 → 下限 1、y 全 1 → 上限 10；第二個：10000000 = 128 → 1 + 128 × 9 / 255

[[ 1.         10.        ]
 [ 5.51764706  5.51764706]]


### GA3_select：競賽式選擇（複製）

```matlab
function result=GA3_select(popu,fitness)
  m=size(popu,1);   %取得popu之列數  即族群中之染色體總數
  for i=1:m
      ind1=1+fix(m*rand); %隨機產生1~m之整數
      ind2=1+fix(m*rand); %隨機產生1~m之整數
      while(ind1==ind2)   %若以上所產生之整數相同則重新隨機產生1~m之整數直到不同為止
       ind2=1+fix(m*rand);
      end
      if fitness(ind1)>=fitness(ind2)   %選擇適應值大的染色體放入交配池 result
          result(i,:)=popu(ind1,:);
      else
          result(i,:)=popu(ind2,:);
      end
  end
```

In [4]:
def GA3_select(popu, fitness):
    m = popu.shape[0]
    result = np.zeros_like(popu)
    for i in range(m):
        ind1 = int(m * rng.random())                  # 0～m−1（MATLAB 是 1～m）
        ind2 = int(m * rng.random())
        while ind1 == ind2:                           # 兩個相同就重抽
            ind2 = int(m * rng.random())
        result[i] = popu[ind1] if fitness[ind1] >= fitness[ind2] else popu[ind2]
    return result

### GA4_Crossover：單點交配

```matlab
function result=GA4_Crossover(popu,co_rate)
   m=size(popu,1);
   n=size(popu,2);
   for i=1:2:m-1
     if rand<co_rate        %co_rate 為交配率
        cross_pt=1+fix((n-1)*rand);  %產生1~(n-1)之隨機整數為交配點
        tmp=popu(i,cross_pt+1:n);   %將交配點後之基因暫存於tmp中
        popu(i,cross_pt+1:n)=popu(i+1,cross_pt+1:n); %以第二個染色體交配點後之基因取代
        popu(i+1,cross_pt+1:n)=tmp; %以暫存於tmp中之基因取代第二個染色體交配點後之基因
     end
   end
   result=popu;
```
MATLAB 的 `cross_pt+1:n`（第 cross_pt+1 到第 n 個基因）在 Python 是 `[cross_pt:]`。

In [5]:
def GA4_Crossover(popu, co_rate):
    popu = popu.copy()
    m, n = popu.shape
    for i in range(0, m - 1, 2):                      # 第 1、2 個一對，第 3、4 個一對……
        if rng.random() < co_rate:
            cross_pt = 1 + int((n - 1) * rng.random())  # 1～n−1
            tmp = popu[i, cross_pt:].copy()
            popu[i, cross_pt:] = popu[i + 1, cross_pt:]
            popu[i + 1, cross_pt:] = tmp
    return popu

a = np.array([[1] * 8, [0] * 8])
print(GA4_Crossover(a, 1.0))                          # 交配率 1：一定交配，看交換了哪幾個基因

[[1 1 0 0 0 0 0 0]
 [0 0 1 1 1 1 1 1]]


### GA5_mutate：突變

```matlab
function result=GA5_mutate(popu,mu_rate)
  m=size(popu,1);
  n=size(popu,2);
  mask=rand(m,n)<mu_rate;
  result=xor(popu,mask);
```

In [6]:
def GA5_mutate(popu, mu_rate):
    mask = rng.random(popu.shape) < mu_rate           # 每個位元以 mu_rate 的機率翻轉
    return np.logical_xor(popu, mask).astype(int)

### GA21_Elit1、GA51_Elit2：菁英保留

```matlab
>> f=[10 50 30 20];
>> [fs,ind]=sort(f)
fs  = 10  20  30  50
ind =  1   4   3   2
>> maxf=f(ind(end))      % 最後一個就是最大值的位置
```
```matlab
function bestxy=GA21_Elit1(popu,fitness,elit_no)
PopuSize=size(popu,1);
[fit,ind]=sort(fitness);			%排序(fit是fitness的值, ind是fitness的指標)
clear fit;
for ii=1:elit_no
 bestxy(ii,:)=popu(ind(PopuSize+1-ii),:);
end
```
```matlab
function result=GA51_Elit2(popu,bestxy)
elit_no=size(bestxy,1);
for ii=1:elit_no
 popu(ii,:)=bestxy(ii,:);
end
result=popu;
```

In [7]:
f = np.array([10, 50, 30, 20])
ind = np.argsort(f, kind="stable")
print("fs  =", f[ind])
print("ind =", ind + 1, "（加 1 換成 MATLAB 的編號）")
print("maxf =", f[ind[-1]])

def GA21_Elit1(popu, fitness, elit_no):
    ind = np.argsort(fitness, kind="stable")          # 由小到大
    return np.array([popu[ind[-1 - ii]] for ii in range(elit_no)])   # 由最大的往回取

def GA51_Elit2(popu, bestxy):
    popu = popu.copy()
    popu[:len(bestxy)] = bestxy                       # 放回族群的前 elit_no 個
    return popu

fs  = [10 20 30 50]
ind = [1 4 3 2] （加 1 換成 MATLAB 的編號）
maxf = 50


## 2. 範例一 ga1.m：求 x² 的最大值（2 ≤ x ≤ 10）

整個流程寫在同一個檔案，沒有呼叫上面的函式：

```matlab
clear all;
chron=20;
gn=8;
generation=50;
xmax=10;
xmin=2;
crate=0.8;
mrate=0.2;
popu=rand(chron,gn)>0.5;

for jj=1:generation
for ii=1:chron
  x(ii,1)=xmin+(xmax-xmin)/(2^gn-1)*polyval(popu(ii,:),2);
  fit(ii,1)=x(ii,1)^2;
end
% 保留菁英
  [bestfit idmax]=max(fit);
  bestchrom=popu(idmax,:);
%複製----------
  for ii=1:chron
      ind1=1+fix(chron*rand); %隨機產生1~m之整數
      ind2=1+fix(chron*rand); %隨機產生1~m之整數
      while(ind1==ind2)
       ind2=1+fix(chron*rand);
      end
      if fit(ind1)>=fit(ind2)   %選擇適應值大的染色體放入交配池 popunew
          popunew(ii,:)=popu(ind1,:);
      else
           popunew(ii,:)=popu(ind2,:);
      end
  end
%交配
for ii=1:2:chron-1
     if rand<crate        %crate 為交配率
        cross_pt=1+fix((gn-1)*rand);  %產生1~(gn-1)之隨機整數為交配點
        tmp=popunew(ii,cross_pt+1:gn);
        popunew(ii,cross_pt+1:gn)=popunew(ii+1,cross_pt+1:gn);
        popunew(ii+1,cross_pt+1:gn)=tmp;
     end
end
%突變
mask=rand(chron,gn)<mrate;
 popunew=xor(popunew,mask);
%精英取代
popunew(1,:)=bestchrom;
popu=popunew;
end

for ii=1:chron
  x(ii,1)=xmin+(xmax-xmin)/(2^gn-1)*polyval(popu(ii,:),2);
  fit(ii,1)=x(ii,1)^2;
end
[fitbest id]=max(fit);
  bestchrom=popu(id,:);
  xbest=x(id,1)
```

In [8]:
rng = np.random.default_rng(1)
chron, gn, generation = 20, 8, 50
xmax, xmin = 10, 2
crate, mrate = 0.8, 0.2
popu = (rng.random((chron, gn)) > 0.5).astype(int)

def decode_x(popu):
    return xmin + (xmax - xmin) / (2 ** gn - 1) * np.array([np.polyval(c, 2) for c in popu])

history = []                                          # 補充：記錄每一代的最大適應值，方便畫圖
for jj in range(generation):
    x = decode_x(popu)
    fit = x ** 2
    idmax = int(np.argmax(fit))                       # 保留菁英
    bestchrom = popu[idmax].copy()
    history.append(fit.max())
    popunew = np.zeros_like(popu)                     # 複製（競賽式選擇）
    for ii in range(chron):
        ind1 = int(chron * rng.random()); ind2 = int(chron * rng.random())
        while ind1 == ind2:
            ind2 = int(chron * rng.random())
        popunew[ii] = popu[ind1] if fit[ind1] >= fit[ind2] else popu[ind2]
    for ii in range(0, chron - 1, 2):                 # 交配
        if rng.random() < crate:
            cross_pt = 1 + int((gn - 1) * rng.random())
            tmp = popunew[ii, cross_pt:].copy()
            popunew[ii, cross_pt:] = popunew[ii + 1, cross_pt:]
            popunew[ii + 1, cross_pt:] = tmp
    mask = rng.random((chron, gn)) < mrate            # 突變
    popunew = np.logical_xor(popunew, mask).astype(int)
    popunew[0] = bestchrom                            # 菁英取代
    popu = popunew

x = decode_x(popu)
fit = x ** 2
xbest = x[int(np.argmax(fit))]
print("xbest =", xbest, "，染色體", popu[int(np.argmax(fit))])

fig = go.Figure(go.Scatter(x=list(range(1, generation + 1)), y=history, mode="lines+markers"))
fig.update_layout(title="ga1.m: maximize x^2 on [2, 10]", xaxis_title="generation", yaxis_title="best fitness x^2", height=320)
fig.show()

xbest = 10.0 ，染色體 [1 1 1 1 1 1 1 1]


突變率 0.2 很高（每代平均每個染色體翻轉 1.6 個位元），族群一直很亂，但因為每代都把最好的染色體放回第一個，最佳值不會變差，很快就找到 x = 10（染色體全為 1）。

## 3. 範例 GAExample.m：只做一代，看每一步的效果

用前面七個函式組成一次完整的演化，求 F(x, y) = x² + y² 在 1 ≤ x, y ≤ 10 的最大值：

```matlab
clear
PopuSize=20;                %族群數
bit_n=8;                   %位元數
var_n=2;                    %變數個數
Pc=0.8;						%交配率
Pm=0.1;						%突變率
n=bit_n*var_n;              %染色體總位元數
range=[1 1; ...
        10  10];            %兩個變數上下限
gen_no=20;                 %預定迭代次數
elit_no=2;
popu=GA1_InitPopu(PopuSize,bit_n,var_n);
xy=GA2_Decode(popu,range) ;
x=xy(:,1);
y=xy(:,2);
fxy=x.^2+y.^2;      %即為: F(x,y) 我們要求最大值的函數
fitness=fxy
bestxy=GA21_Elit1(popu,fitness,elit_no);
popu=GA3_select(popu,fitness);
popu=GA4_Crossover(popu,Pc) ;
popu=GA5_mutate(popu,Pm);
popu=GA51_Elit2(popu,bestxy);
%重複以上步驟
xy=GA2_Decode(popu,range);
x=xy(:,1);
y=xy(:,2);
fxy=x.^2+y.^2;
fitness=fxy
```

In [9]:
rng = np.random.default_rng(2)
PopuSize, bit_n, var_n, Pc, Pm, elit_no = 20, 8, 2, 0.8, 0.1, 2
range_ = np.array([[1, 1], [10, 10]])

popu = GA1_InitPopu(PopuSize, bit_n, var_n)
xy = GA2_Decode(popu, range_)
fitness0 = xy[:, 0] ** 2 + xy[:, 1] ** 2
bestxy = GA21_Elit1(popu, fitness0, elit_no)
popu = GA3_select(popu, fitness0)
popu = GA4_Crossover(popu, Pc)
popu = GA5_mutate(popu, Pm)
popu = GA51_Elit2(popu, bestxy)
xy = GA2_Decode(popu, range_)
fitness1 = xy[:, 0] ** 2 + xy[:, 1] ** 2
print("第 0 代：平均 %.1f，最大 %.1f" % (fitness0.mean(), fitness0.max()))
print("第 1 代：平均 %.1f，最大 %.1f" % (fitness1.mean(), fitness1.max()))
print("第 1 代的前 2 個是保留下來的菁英：", np.round(fitness1[:2], 2))

第 0 代：平均 74.9，最大 174.1
第 1 代：平均 111.9，最大 174.1
第 1 代的前 2 個是保留下來的菁英： [174.09 151.18]


選擇讓族群的平均適應值上升；菁英保留保證最大值不會下降。

## 4. 範例 GAExample1.m：演化 20 代

```matlab
clear
PopuSize=40;                %族群數
bit_n=8;                   %位元數
var_n=2;                    %變數個數
Pc=0.8;						%交配率
Pm=0.1;						%突變率
n=bit_n*var_n;              %染色體總位元數
range=[1 2; ...
        5  10];            %兩個變數上下限
gen_no=20;                 %預定迭代次數
elit_no=2;
popu=GA1_InitPopu(PopuSize,bit_n,var_n);
for jj=1:gen_no
   xy=GA2_Decode(popu,range) ;
   x=xy(:,1);
   y=xy(:,2);
   fxy=x.^2+y.^2;      %即為: F(x,y) 我們要求最大值的函數
   fitness=fxy;
   bestxy=GA21_Elit1(popu,fitness,elit_no);
   popu=GA3_select(popu,fitness);
   popu=GA4_Crossover(popu,Pc) ;
   popu=GA5_mutate(popu,Pm);
   popu=GA51_Elit2(popu,bestxy);
end
xy=GA2_Decode(popu,range);
x=xy(:,1);
y=xy(:,2);
fxy=x.^2+y.^2;
fitness=fxy;
[fit,ind]=sort(fitness);			%排序(fit是fitness的值, ind是fitness的指標)
clear fit;
bestxy=xy(ind(PopuSize),:)
```

In [10]:
rng = np.random.default_rng(3)
PopuSize, bit_n, var_n, Pc, Pm, gen_no, elit_no = 40, 8, 2, 0.8, 0.1, 20, 2
range_ = np.array([[1, 2], [5, 10]])                  # x 在 1～5、y 在 2～10

popu = GA1_InitPopu(PopuSize, bit_n, var_n)
for jj in range(gen_no):
    xy = GA2_Decode(popu, range_)
    fitness = xy[:, 0] ** 2 + xy[:, 1] ** 2
    bestxy = GA21_Elit1(popu, fitness, elit_no)
    popu = GA3_select(popu, fitness)
    popu = GA4_Crossover(popu, Pc)
    popu = GA5_mutate(popu, Pm)
    popu = GA51_Elit2(popu, bestxy)

xy = GA2_Decode(popu, range_)
fitness = xy[:, 0] ** 2 + xy[:, 1] ** 2
ind = np.argsort(fitness, kind="stable")
bestxy = xy[ind[-1]]
print("bestxy =", bestxy, "，F =", fitness.max(), "（理論最大值 5² + 10² = 125）")

bestxy = [ 5. 10.] ，F = 125.0 （理論最大值 5² + 10² = 125）


## 5. 補充：多峰函數 f(x, y) = x sin(4x) + 1.1 y sin(y)

[適應函數](https://yoshiagent.github.io/IntControl/GA-FitnessFunction.html)一頁的函數有許多分開的峰。用同樣的七個函式（16 位元、50 個染色體、100 代）找最大值，再和細格點搜尋的真正最大值比較：

In [11]:
rng = np.random.default_rng(4)
fxy = lambda x, y: x * np.sin(4 * x) + 1.1 * y * np.sin(y)
range_ = np.array([[0, 0], [10, 10]])
popu = GA1_InitPopu(50, 16, 2)
for g in range(100):
    xy = GA2_Decode(popu, range_)
    fitness = fxy(xy[:, 0], xy[:, 1])
    bestxy = GA21_Elit1(popu, fitness, 2)
    popu = GA51_Elit2(GA5_mutate(GA4_Crossover(GA3_select(popu, fitness), 0.8), 0.02), bestxy)
xy = GA2_Decode(popu, range_)
fitness = fxy(xy[:, 0], xy[:, 1])
k = int(np.argmax(fitness))
g1 = np.linspace(0, 10, 2001)
gx, gy = np.meshgrid(g1, g1)
gf = fxy(gx, gy)
print("GA 找到：x = %.4f, y = %.4f, f = %.4f" % (xy[k, 0], xy[k, 1], fitness[k]))
print("格點搜尋：x = %.4f, y = %.4f, f = %.4f" % (gx.flat[gf.argmax()], gy.flat[gf.argmax()], gf.max()))

fig = go.Figure(go.Contour(x=g1[::20], y=g1[::20], z=np.round(gf[::20, ::20], 3), colorscale="Viridis", colorbar_title="f"))
fig.add_scatter(x=xy[:, 0], y=xy[:, 1], mode="markers", marker=dict(color="white", size=6), name="final population")
fig.add_scatter(x=[xy[k, 0]], y=[xy[k, 1]], mode="markers", marker=dict(color="red", size=14, symbol="star"), name="GA best")
fig.update_layout(title="x sin(4x) + 1.1 y sin(y)", xaxis_title="x", yaxis_title="y", width=560, height=480)
fig.show()

GA 找到：x = 9.8239, y = 7.9785, f = 18.5291

格點搜尋：x = 9.8250, y = 7.9800, f = 18.5289


## 6. 範例 MCKPIDGA_Main.m：用 GA 調 PID

原講義附檔 gamckpid7.zip。染色體是 PID 的三個增益（各 8 位元、0.1～10），每個染色體都用 Simulink 模型 `mckpidgamdl.mdl` 模擬一次 MCK 系統（m = 1、c = 1、k = 2，取自模型的遮罩參數）的單位步階響應 10 秒，成本是 1.2 秒之後的誤差平方和、超越 15% 再加罰 1000，適應值 = 3000 − 成本：

```matlab
clear
Min_Offset=3000;
Elite_No=2;
MDL='MCKPIDGAmdl';
range = [0.1  0.1  0.1; ...
          10     10      10];
gen_no=20;
PopuSize=10;
bit_n=8;
var_n=size(range,2);
xover_rate=0.9;
mutate_rate=0.03;
tf=10;
popu=GA1_Initpopu(PopuSize,var_n,bit_n);
bestkpid=zeros(gen_no,3);   % for recording best KpKiKd in each generation
bestfit=zeros(gen_no,1);    % for recording best fitness in each generation
for j=1:gen_no   % for each generation begin
  kpkikd=GA2_decode(popu,range);
  for i=1:PopuSize
    P=kpkikd(i,1);
    I=kpkikd(i,2);
    D=kpkikd(i,3);
    sim(MDL,[0 tf]);   %run simulink
    ti=find(tout>1.2);
    err=(in(ti:size(in,1))-x(ti:size(x,1)))'*(in(ti:size(in,1))-x(ti:size(x,1)));
    if max(x)>1.15               % maximum overshoot requirement
        err=err+1000;     %% if the max-value of x is larger then 1.15, PI will be increased!
     end
    fitness(i)=Min_Offset-err;     %%Let the fitness show the maximum
  end
  [fit, ind]=sort(fitness);  %from small to big
  clear fit;
  for ii=1:Elite_No         %將最佳染色體保留
    ElitePopu(ii,:)=popu(ind(PopuSize-ii+1),:);
  end
  bestkpid(j,:)=kpkikd(ind(PopuSize),:);
  bestfit(j,1)=fitness(ind(PopuSize));
  popu=GA3_select(popu,fitness);
  popu=GA4_Crossover(popu,xover_rate);
  popu=GA5_mutate(popu,mutate_rate);
   for ii=1:Elite_No     %將保留之最佳染色體置入
    popu(PopuSize-ii+1,:)=ElitePopu(ii,:);
  end
end   % for each generation end
    [best,ind1]=sort(bestfit);
    clear best;
    P=bestkpid(ind1(gen_no),1)
    I=bestkpid(ind1(gen_no),2)
    D=bestkpid(ind1(gen_no),3)
      g=1:1:gen_no;
    sim(MDL,[0 tf]);
    plot(g,bestfit,'o',g,bestfit,'-');   %plot the best-fitness of each generation
    grid;
```

和前面不同的地方：

* 這個附檔裡的 `GA4_Crossover.m` 是另一個版本：交配點 `cross_pt=2+fix((n-1)*rand)`（2～n），交換第 cross_pt 到第 n 個基因。
* 菁英放回族群的**最後**兩個位置（前面的範例放在最前面）。
* `GA1_Initpopu(PopuSize,var_n,bit_n)` 的後兩個引數順序和函式定義相反，但函式只用到兩者的乘積，結果相同。

**Simulink 模型改寫成 Python**：`mckpidgamdl.mdl` 的結構是 步階（t = 0 由 0 跳到 1）→ 誤差 e = r − x → PID 子系統（`P·e`、`I·∫e`、`d/dt(D·e)` 相加）→ MCK 子系統（ẍ = (u − c ẋ − k x)/m）。下面用固定步長 0.01 s 的四階 Runge–Kutta 積分。設定值固定時 de/dt = −ẋ。
原模型的求解器設定為 ode5、固定步長自動（`sim(MDL,[0 tf])` 時為 10/50 = 0.2 s），所以 `in`、`x` 每 0.2 s 存一點，誤差平方和也照這個取樣間隔計算。Simulink 的 Derivative 方塊用前後兩步的差分近似微分，和這裡的 −ẋ 會有些微差異。

In [12]:
m_, c_, k_ = 1.0, 1.0, 2.0                            # MCK 子系統的遮罩參數 m|c|k = 1|1|2

def mck_pid(P, I, D, tf=10.0, dt=0.01, every=20):
    """回傳 Simulink 輸出時間點（每 every·dt = 0.2 s）的 t、r、x"""
    def f(s):
        x, v, z = s                                   # z：積分器的輸出 ∫ I·e dt
        e = 1.0 - x
        u = P * e + z + D * (-v)                      # d/dt(D·e) = −D·ẋ
        return np.array([v, (u - c_ * v - k_ * x) / m_, I * e])
    s = np.zeros(3)
    ts, xs = [0.0], [0.0]
    for i in range(1, int(round(tf / dt)) + 1):
        k1 = f(s); k2 = f(s + dt / 2 * k1); k3 = f(s + dt / 2 * k2); k4 = f(s + dt * k3)
        s = s + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        if i % every == 0:
            ts.append(i * dt); xs.append(s[0])
    return np.array(ts), np.ones(len(ts)), np.array(xs)

def fitness_pid(P, I, D, Min_Offset=3000):
    t, r, x = mck_pid(P, I, D)
    ti = t > 1.2
    err = float((r[ti] - x[ti]) @ (r[ti] - x[ti]))
    if x.max() > 1.15:                                # 超越量大於 15% 加罰
        err += 1000
    return Min_Offset - err

def GA4_Crossover_v2(popu, co_rate):
    """gamckpid7.zip 裡的版本：交配點 2～n（MATLAB 編號），交換第 cross_pt～n 個基因"""
    popu = popu.copy()
    m, n = popu.shape
    for i in range(0, m - 1, 2):
        if rng.random() < co_rate:
            cross_pt = 2 + int((n - 1) * rng.random())  # MATLAB 編號 2～n
            tmp = popu[i, cross_pt - 1:].copy()
            popu[i, cross_pt - 1:] = popu[i + 1, cross_pt - 1:]
            popu[i + 1, cross_pt - 1:] = tmp
    return popu

In [13]:
rng = np.random.default_rng(5)
Elite_No, gen_no, PopuSize, bit_n = 2, 20, 10, 8
range_ = np.array([[0.1, 0.1, 0.1], [10, 10, 10]])
xover_rate, mutate_rate = 0.9, 0.03
var_n = range_.shape[1]

popu = GA1_InitPopu(PopuSize, bit_n, var_n)
bestkpid, bestfit = np.zeros((gen_no, 3)), np.zeros(gen_no)
for j in range(gen_no):
    kpkikd = GA2_Decode(popu, range_)
    fitness = np.array([fitness_pid(*g) for g in kpkikd])
    ind = np.argsort(fitness, kind="stable")
    ElitePopu = np.array([popu[ind[PopuSize - 1 - ii]] for ii in range(Elite_No)])
    bestkpid[j], bestfit[j] = kpkikd[ind[-1]], fitness[ind[-1]]
    popu = GA3_select(popu, fitness)
    popu = GA4_Crossover_v2(popu, xover_rate)
    popu = GA5_mutate(popu, mutate_rate)
    for ii in range(Elite_No):
        popu[PopuSize - 1 - ii] = ElitePopu[ii]

P, I, D = bestkpid[int(np.argsort(bestfit, kind="stable")[-1])]
print(f"P = {P:.4f}, I = {I:.4f}, D = {D:.4f}，適應值 {bestfit.max():.4f}")

t, r, x = mck_pid(P, I, D)
fig = make_subplots(rows=1, cols=2, subplot_titles=("best fitness of each generation", f"step response, P={P:.2f}, I={I:.2f}, D={D:.2f}"))
fig.add_scatter(x=list(range(1, gen_no + 1)), y=bestfit, mode="lines+markers", name="best fitness", row=1, col=1)
fig.add_scatter(x=t, y=r, mode="lines", line=dict(dash="dash", color="gray"), name="set point", row=1, col=2)
fig.add_scatter(x=t, y=x, mode="lines+markers", name="x (every 0.2 s)", row=1, col=2)
fig.update_xaxes(title_text="generation", row=1, col=1); fig.update_xaxes(title_text="t (s)", row=1, col=2)
fig.update_layout(height=360)
fig.show()

P = 9.9224, I = 5.1082, D = 3.4388，適應值 2999.9894


適應值最大是 3000（誤差平方和為 0）。增益範圍只到 10，GA 通常會把 P、D 推到接近上限：增益越大，響應越快、誤差平方和越小，而 D 提供阻尼，讓超越量不超過 15%。

## 7. 範例 MCKFuzzyGa_Main.m：用 GA 調模糊控制器的比例因子

原講義附檔 MCKGaFuzzy.zip。控制器是 Mamdani 模糊控制器 `GaFuzzy2.fis`（輸入誤差 e 與誤差變化率 de、各 7 個歸屬函數、12 條規則），GA 調整的是三個**比例因子**：輸入乘上 Se、Sde（再限制在 ±10），輸出乘上 Su。主程式和上一個幾乎相同，差別是：

```matlab
Min_Offset=5000;
MDL='GaFuzzy';
range = [0.1  0.005  0.001; ...
          5     1      10];
...
    Se=kpkikd(i,1);
    Sde=kpkikd(i,2);
    Su=kpkikd(i,3);
    sim(MDL,[0 tf]);   %run simulink
    ti=find(tout>1.2);
    err=abs(in(ti:end)-x(ti:end)).^2;
    err=sum(err);         %% let err = every element of err being summed
    if max(x)>1.15
        err=err+1000;
    end
    fitness(i)=Min_Offset-err;
```

模型 `GaFuzzy.mdl` 的求解器是 ode5、固定步長 0.01 s，所以誤差每 0.01 s 取一點。下面先寫一個小型的 .fis 讀檔與 Mamdani 推論（min–max、重心法，輸出範圍切 101 點，和 MATLAB Fuzzy Logic Toolbox 相同；輸入超出範圍時以範圍邊界計算）。

In [14]:
GAFUZZY2 = """[System]
Name='GaFuzzy2'
Type='mamdani'
NumInputs=2
NumOutputs=1
NumRules=12
AndMethod='min'
OrMethod='max'
ImpMethod='min'
AggMethod='max'
DefuzzMethod='centroid'

[Input1]
Name='error'
Range=[-1 1]
NumMFs=7
MF1='NS':'trimf',[-0.6 -0.3 0]
MF2='ZR':'trimf',[-0.3 0 0.3]
MF3='PS':'trimf',[0 0.3 0.6]
MF4='NL':'trapmf',[-1 -1 -0.9 -0.6]
MF5='PL':'trapmf',[0.6 0.9 1 1]
MF6='PM':'trimf',[0.3 0.6 0.9]
MF7='NM':'trimf',[-0.9 -0.6 -0.3]

[Input2]
Name='derror'
Range=[-1 1]
NumMFs=7
MF1='NL':'trapmf',[-1 -1 -0.9 -0.6]
MF2='NS':'trimf',[-0.6 -0.3 0]
MF3='ZR':'trimf',[-0.3 0 0.3]
MF4='PS':'trimf',[0 0.3 0.6]
MF5='PL':'trapmf',[0.6 0.9 1 1]
MF6='PM':'trimf',[0.3 0.6 0.9]
MF7='NM':'trimf',[-0.9 -0.6 -0.3]

[Output1]
Name='output1'
Range=[-5 5]
NumMFs=7
MF1='NL':'trapmf',[-5 -5 -4.5 -3]
MF2='Z':'trimf',[-1.5 0 1.5]
MF3='PL':'trapmf',[3 4.5 5 5]
MF4='NS':'trimf',[-3 -1.5 0]
MF5='PS':'trimf',[0 1.5 3]
MF6='NM':'trimf',[-4.5 -3 -1.5]
MF7='PM':'trimf',[1.5 3 4.5]

[Rules]
5 3, 3 (1) : 1
6 3, 3 (1) : 1
3 3, 3 (1) : 1
5 2, 3 (1) : 1
5 7, 3 (1) : 1
5 1, 3 (1) : 1
3 4, 5 (1) : 1
3 6, 7 (1) : 1
3 5, 3 (1) : 1
2 4, 5 (1) : 1
1 0, 6 (1) : 1
3 1, 6 (1) : 1
"""

def mf(kind, p, x):
    x = np.asarray(x, float)
    if kind == "trimf":
        a, b, c = p
        l = (x - a) / (b - a) if b > a else (x >= b) * 1.0
        r = (c - x) / (c - b) if c > b else (x <= b) * 1.0
        return np.clip(np.minimum(l, r), 0, 1)
    if kind == "trapmf":
        a, b, c, d = p
        l = (x - a) / (b - a) if b > a else (x >= b) * 1.0
        r = (d - x) / (d - c) if d > c else (x <= c) * 1.0
        return np.clip(np.minimum(np.minimum(l, 1.0), r), 0, 1)
    raise ValueError(kind)

def readfis_text(txt):
    """把 .fis 的文字解析成 dict（只處理本筆記本用到的欄位）"""
    fis, sec, cur = {"inputs": [], "outputs": [], "rules": []}, None, None
    for line in txt.splitlines():
        line = line.strip()
        if not line:
            continue
        mm = re.match(r"\[(\w+?)(\d*)\]", line)
        if mm:
            sec = mm.group(1)
            if sec in ("Input", "Output"):
                cur = {"mfs": []}
                fis["inputs" if sec == "Input" else "outputs"].append(cur)
            continue
        if sec == "Rules":
            a, rest = line.split(",")
            c, wt, conn = re.match(r"\s*([-\d\s]+)\(([\d.]+)\)\s*:\s*(\d)", rest).groups()
            fis["rules"].append(([int(v) for v in a.split()], [int(v) for v in c.split()], float(wt), int(conn)))
            continue
        key, val = (s.strip() for s in line.split("=", 1))
        if sec == "System":
            fis[key] = val.strip("'")
        elif key == "Range":
            cur["range"] = [float(v) for v in val.strip("[]").split()]
        elif key.startswith("MF"):
            name, kind, p = re.match(r"'(.*?)':'(\w+)',\[(.*)\]", val).groups()
            cur["mfs"].append((name, kind, [float(v) for v in p.split()]))
    return fis

def evalfis_mamdani(x, fis, npts=101):
    """Mamdani：AND 取 min、截頂（min）、聚合取 max、重心法解模糊化"""
    x = [min(max(v, lo), hi) for v, (lo, hi) in zip(x, [i["range"] for i in fis["inputs"]])]
    lo, hi = fis["outputs"][0]["range"]
    t = np.linspace(lo, hi, npts)
    agg = np.zeros(npts)
    for ante, cons, wt, conn in fis["rules"]:
        degs = [float(mf(*fis["inputs"][i]["mfs"][a - 1][1:], x[i])) for i, a in enumerate(ante) if a != 0]
        w = wt * (min(degs) if conn == 1 else max(degs))
        agg = np.maximum(agg, np.minimum(w, mf(*fis["outputs"][0]["mfs"][cons[0] - 1][1:], t)))
    return float(np.sum(agg * t) / np.sum(agg)) if agg.sum() > 0 else (lo + hi) / 2

gafis = readfis_text(GAFUZZY2)
print("evalfis([0.5 0], GaFuzzy2) =", round(evalfis_mamdani([0.5, 0.0], gafis), 4))
print("evalfis([0 0], GaFuzzy2)   =", round(evalfis_mamdani([0.0, 0.0], gafis), 4))

evalfis([0.5 0], GaFuzzy2) = 4.2484
evalfis([0 0], GaFuzzy2)   = 0.0


In [15]:
def mck_fuzzy(Se, Sde, Su, tf=10.0, dt=0.01):
    """GaFuzzy.mdl：e → Se → 限制 ±10、de/dt → Sde → 限制 ±10 → 模糊控制器 → Su → MCK。
    控制量在每個 0.01 s 步長開始時計算一次（零階保持），MCK 用 RK4 積分"""
    x = v = 0.0
    ts, xs = [0.0], [0.0]
    for i in range(int(round(tf / dt))):
        e, de = 1.0 - x, -v
        u = Su * evalfis_mamdani([np.clip(Se * e, -10, 10), np.clip(Sde * de, -10, 10)], gafis)
        f = lambda s: np.array([s[1], (u - c_ * s[1] - k_ * s[0]) / m_])
        s = np.array([x, v])
        k1 = f(s); k2 = f(s + dt / 2 * k1); k3 = f(s + dt / 2 * k2); k4 = f(s + dt * k3)
        x, v = s + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        ts.append((i + 1) * dt); xs.append(x)
    return np.array(ts), np.ones(len(ts)), np.array(xs)

def fitness_fuzzy(Se, Sde, Su, Min_Offset=5000):
    t, r, x = mck_fuzzy(Se, Sde, Su)
    ti = t > 1.2
    err = float(np.sum(np.abs(r[ti] - x[ti]) ** 2))
    if x.max() > 1.15:
        err += 1000
    return Min_Offset - err

In [16]:
rng = np.random.default_rng(6)
range_ = np.array([[0.1, 0.005, 0.001], [5, 1, 10]])
popu = GA1_InitPopu(PopuSize, bit_n, 3)
bestS, bestfitF = np.zeros((gen_no, 3)), np.zeros(gen_no)
for j in range(gen_no):
    S = GA2_Decode(popu, range_)
    fitness = np.array([fitness_fuzzy(*s) for s in S])
    ind = np.argsort(fitness, kind="stable")
    ElitePopu = np.array([popu[ind[PopuSize - 1 - ii]] for ii in range(Elite_No)])
    bestS[j], bestfitF[j] = S[ind[-1]], fitness[ind[-1]]
    popu = GA5_mutate(GA4_Crossover_v2(GA3_select(popu, fitness), xover_rate), mutate_rate)
    for ii in range(Elite_No):
        popu[PopuSize - 1 - ii] = ElitePopu[ii]

Se, Sde, Su = bestS[int(np.argsort(bestfitF, kind="stable")[-1])]
t, r, x = mck_fuzzy(Se, Sde, Su)
print(f"Se = {Se:.4f}, Sde = {Sde:.4f}, Su = {Su:.4f}，適應值 {bestfitF.max():.4f}"
      f"（1.2 秒後的誤差平方和 {5000 - bestfitF.max():.2e}，最大值 x = {x.max():.4f}）")
tp, rp, xp = mck_pid(P, I, D, every=1)
fig = make_subplots(rows=1, cols=2, subplot_titles=("best fitness of each generation", "step response"))
fig.add_scatter(x=list(range(1, gen_no + 1)), y=bestfitF, mode="lines+markers", name="best fitness", row=1, col=1)
fig.add_scatter(x=t, y=r, mode="lines", line=dict(dash="dash", color="gray"), name="set point", row=1, col=2)
fig.add_scatter(x=t, y=x, mode="lines", name="fuzzy (GA-tuned)", row=1, col=2)
fig.add_scatter(x=tp, y=xp, mode="lines", name="PID (GA-tuned)", row=1, col=2)
fig.update_xaxes(title_text="generation", row=1, col=1); fig.update_xaxes(title_text="t (s)", row=1, col=2)
fig.update_layout(height=360)
fig.show()

Se = 1.4835, Sde = 0.6683, Su = 3.1379，適應值 4999.9982（1.2 秒後的誤差平方和 1.85e-03，最大值 x = 1.0816）


注意這裡的適應值（5000 − 誤差平方和，每 0.01 s 一點）和 PID 範例（3000 − 誤差平方和，每 0.2 s 一點）的尺度不同，不能直接比大小；要比較兩個控制器，看右圖的響應即可。

想進一步了解：網站的 [GA 章](https://yoshiagent.github.io/IntControl/GA.html)有互動的演化模擬；
[現代最佳化與調參](https://yoshiagent.github.io/IntControl/OPT.html)一章比較了 GA、PSO、CMA-ES 與貝氏最佳化調 PID 的效果。